# Import Statements

In [ ]:
import os
import custom_cmap
from PIL import Image
from IPython.display import display

import matplotlib.pyplot as plt
import matplotlib as mpl
import pandas as pd

from moria import reduce
from pathlib import Path

from astropy.visualization import PercentileInterval
from astropy.io import fits
from astropy.visualization import LogStretch, ImageNormalize
import plotly.express as px
import numpy as np
from astropy.visualization import ImageNormalize, AsinhStretch, SqrtStretch, LogStretch, PowerStretch

plt.rcParams.update({'font.size':25})
plt.rc('text', usetex=True)
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2

# Understanding the main directory. 

The directory structure for your data analysis should be organized as follows. 

<li>00.DATA</li>
<li>01.XYM</li>
<li>02.CMD</li>
<li>03.LOC_TRANS</li>
<li>04.PSF_EXTRACT</li>
<li>05.COORD_TRANS(OPTIONAL)</li>
<li>06.FIT</li>
<li>07.CALIBRATION</li>

This directory structure has been set up in "MORIA/data". 

All necessary scripts are included within the corresponding folders under MORIA/data. Therefore, the simplest way to run MORIA on your target is to copy all eight folders from MORIA/data to the location where you intend to perform your analysis. Begin by placing your exposures in the corresponding data/00.DATA/ directories (e.g. the /F814W and /F606W filter sub-directories).

NOTE: chmod +x program.src is a useful command to use whenever permissions are denied for a script. If any of these scripts fail, each folder in the list above (00.DATA, 01.XYM, 02.CMD, etc.) has an output "log file" for the Fortran scripts we run. If the "log file" for the step you ran shows a "permission denied" error, it is very likely that you need to run chmod _x program.src for each ".src" file used in this pipeline.

# Notebook Goal

After you finish running this notebook, you will be able to use the "_flc" exposures to generate an output image stack in the F814W and F606W HST filters.

This notebook will prepare files in 00.DATA, 01.XYM and 02.CMD.

In [ ]:
#This is the directory where you are processing your data. This does not point at MORIA.
directory = os.getcwd()

# Data Preparation 

When you installed MORIA, several fortran scripts were compiled directly in "MORIA/src/fortran_compile". This step copies the fortran files into the correct directories. You are also required to select the camera you want to use MORIA on, i.e., you either use the WFC3UV camera or the ACS camera. In this step, we will also download the "library" PSF files used in step 2.

In [ ]:
camera = 'WFC3UV'
#camera = 'ACS'

In [ ]:
reduce.data_prep_early(camera, directory)

# STEP 1

<li> The cell below will convert the _flc files to _WJ2 files using run_convert_C1K1C.src</li>

These "_flc" exposures include the charge-transfer efficiency (CTE) corrections. MORIA first converts these "_flc" exposures into "_WJ2" files to convert to a full-chip coordiante system that are placed into appropriate reference geometry for subsequent steps. 

In [ ]:
reduce.run_xgf_conversion(directory)



# Step 2 

Prepare the IN.* files that will be read by the scripts in 01.XYM and 02.CMD directories. 

This step is important since the IN.* files in each directory are used to provide input to the backend Fortran scripts.

In [ ]:
reduce.data_prep(directory)

# Create MATCHUP files

In the cell below, we will achieve the following objectives:

1. Use the corresponding PSF files for each filter to create an .XYM file for every exposure. A library PSF model is used to generate the .xyvieeee files, which contain the stellar positions and magnitudes for each HST exposure. 
2. Keep all stars that are found in at least 80% of the exposures for each filter. 

In [ ]:
reduce.matchup_files(camera, directory)

Once the cell above finishes running, you will have the final MATCHUP files. One is located at 01.XYM/F814W/MATCHUP.F814W.XYM.02 for the F814W filter and the other is at 01.XYM/F606W/MATCHUP.F606W.XYM for the F606W filter. 

We will open the MATCHUP files to inspect them further. 

In [ ]:
filename= Path(directory).resolve()/f"01.XYM/dex_no_gaia_STEP08_A.xyvieeee"

cols = ["xbar", "ybar", "mv", "mi"]
df = pd.read_csv(filename, sep=r"\s+", comment="#", header=None, usecols = [0, 1, 2, 3], names=cols)

In [ ]:
df

From the MATCHUP file opened above, here are the most important things to note:

1. xbar and ybar correspond to the mean x and y star positions (in pixel coordinates) for stars in the HST images.
2. mv is the mean instrumental V magnitude of the stars.
3. mi is the mean instrumental V magnitude of the stars.

In this step we have also created a stack of the HST exposures in the F814W and F606W filters. 

You are now ready to view the output stacks for both filters using DS9.

Alternatively, you can view this output stack in the jupyter notebook by running the cell below. The output stack for the F814W filter and the F606W filters are kept in 01.XYM. We open the output stack for F814W below. You can uncomment the filename_606W line in the cell below to look at the F606W output stack.

The scaling can be adjusted depending on your preference by changing the argument given to "ImageNormalize" in the cell below. Note that this file was too big and had to be scaled in terms of size. The full stack of images is greater than 10,000 pixels and kills the kernel/notebook when it runs. The preferred way of viewing it should still be DS9. 

In [ ]:
##############
# READ FILES #
##############

filename_814W = Path(directory).resolve()/f"01.XYM/dex_no_gaia_STEP10B_F814W.fits"
#filename_606W = Path(directory).resolve()/f"01.XYM/dex_no_gaia_STEP10A_F606W.fits"

##################
# OPEN FITS FILE #
##################
hdul = fits.open(filename_814W)
data = hdul[-1].data
hdul.close()
data = np.nan_to_num(data, nan=0.0, posinf=0.0, neginf=0.0)


##########################
# SCALE THE OUTPUT STACK #
##########################
MAX_PREVIEW_SIDE = 1536
def read_stack_preview(path, max_side=MAX_PREVIEW_SIDE):
    with fits.open(path, memmap=True) as hdul:
        raw = hdul[-1].data
        ny, nx = int(raw.shape[0]), int(raw.shape[1])
        step = max(int(np.ceil(max(ny, nx) / max_side)), 1)
        # Small contiguous copy; stride read from memmap avoids holding the full image in memory.
        preview = np.array(raw[::step, ::step], dtype=np.float32, copy=True)
    preview = np.nan_to_num(preview, nan=0.0, posinf=0.0, neginf=0.0)
    return preview, step, (ny, nx)
preview, stride, full_shape = read_stack_preview(filename_814W)
norm = ImageNormalize(preview, stretch=AsinhStretch(0.001))
scaled = norm(preview)
nonbinary_colors = custom_cmap.mpl_nb()

########
# PLOT #
########
fig, ax = plt.subplots(figsize=(11, 11))
im = ax.imshow(scaled, origin="lower", cmap=nonbinary_colors, interpolation="nearest")
ax.set_title(f"Output Stack (F814W)")
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.02, label="Intensity")
plt.show()